# Debugging Neural Networks Notebook

> Hands-on Build It and Exercises.

## Build It

A diagnostic toolkit that monitors activations, gradients, and loss curves. You will deliberately break a network and use the toolkit to diagnose each problem.

### Step 1: The NetworkDebugger Class

Hooks into a PyTorch model to record activation and gradient statistics per layer.

In [ ]:
```python

import torch

import torch.nn as nn

import math

class NetworkDebugger:

    def __init__(self, model):

        self.model = model

        self.activation_stats = {}

        self.gradient_stats = {}

        self.loss_history = []

        self.lr_losses = []

        self.hooks = []

        self._register_hooks()

    def _register_hooks(self):

        for name, module in self.model.named_modules():

            if isinstance(module, (nn.Linear, nn.Conv2d, nn.ReLU, nn.LeakyReLU)):

                hook = module.register_forward_hook(self._make_activation_hook(name))

                self.hooks.append(hook)

                hook = module.register_full_backward_hook(self._make_gradient_hook(name))

                self.hooks.append(hook)

    def _make_activation_hook(self, name):

        def hook(module, input, output):

            with torch.no_grad():

                out = output.detach().float()

                self.activation_stats[name] = {

                    "mean": out.mean().item(),

                    "std": out.std().item(),

                    "fraction_zero": (out == 0).float().mean().item(),

                    "min": out.min().item(),

                    "max": out.max().item(),

                }

        return hook

    def _make_gradient_hook(self, name):

        def hook(module, grad_input, grad_output):

            if grad_output[0] is not None:

                with torch.no_grad():

                    grad = grad_output[0].detach().float()

                    self.gradient_stats[name] = {

                        "mean": grad.mean().item(),

                        "std": grad.std().item(),

                        "abs_mean": grad.abs().mean().item(),

                        "max": grad.abs().max().item(),

                    }

        return hook

    def record_loss(self, loss_value):

        self.loss_history.append(loss_value)

    def check_loss_health(self):

        if len(self.loss_history) < 2:

            return "NOT_ENOUGH_DATA"

        recent = self.loss_history[-10:]

        if any(math.isnan(v) or math.isinf(v) for v in recent):

            return "NAN_OR_INF"

        if len(self.loss_history) >= 20:

            first_half = sum(self.loss_history[:10]) / 10

            second_half = sum(self.loss_history[-10:]) / 10

            if second_half >= first_half * 0.99:

                return "NOT_DECREASING"

        if len(recent) >= 5:

            diffs = [recent[i+1] - recent[i] for i in range(len(recent)-1)]

            if max(diffs) - min(diffs) > 2 * abs(sum(diffs) / len(diffs)):

                return "OSCILLATING"

        return "HEALTHY"

    def check_activations(self):

        issues = []

        for name, stats in self.activation_stats.items():

            if stats["fraction_zero"] > 0.5:

                issues.append(f"DEAD_NEURONS: {name} has {stats['fraction_zero']:.0%} zero activations")

            if abs(stats["mean"]) > 10:

                issues.append(f"EXPLODING_ACTIVATIONS: {name} mean={stats['mean']:.2f}")

            if stats["std"] < 1e-6:

                issues.append(f"COLLAPSED_ACTIVATIONS: {name} std={stats['std']:.2e}")

        return issues if issues else ["HEALTHY"]

    def check_gradients(self):

        issues = []

        grad_magnitudes = []

        for name, stats in self.gradient_stats.items():

            grad_magnitudes.append((name, stats["abs_mean"]))

            if stats["abs_mean"] < 1e-7:

                issues.append(f"VANISHING_GRADIENT: {name} abs_mean={stats['abs_mean']:.2e}")

            if stats["abs_mean"] > 100:

                issues.append(f"EXPLODING_GRADIENT: {name} abs_mean={stats['abs_mean']:.2e}")

        if len(grad_magnitudes) >= 2:

            first_mag = grad_magnitudes[0][1]

            last_mag = grad_magnitudes[-1][1]

            if last_mag > 0 and first_mag / last_mag > 100:

                issues.append(f"GRADIENT_RATIO: first/last = {first_mag/last_mag:.0f}x (vanishing)")

        return issues if issues else ["HEALTHY"]

    def print_report(self):

        print("\n=== NETWORK DEBUGGER REPORT ===")

        print(f"\nLoss health: {self.check_loss_health()}")

        if self.loss_history:

            print(f"  Last 5 losses: {[f'{v:.4f}' for v in self.loss_history[-5:]]}")

        print("\nActivation diagnostics:")

        for item in self.check_activations():

            print(f"  {item}")

        print("\nGradient diagnostics:")

        for item in self.check_gradients():

            print(f"  {item}")

        print("\nPer-layer activation stats:")

        for name, stats in self.activation_stats.items():

            print(f"  {name}: mean={stats['mean']:.4f} std={stats['std']:.4f} zero={stats['fraction_zero']:.1%}")

        print("\nPer-layer gradient stats:")

        for name, stats in self.gradient_stats.items():

            print(f"  {name}: abs_mean={stats['abs_mean']:.2e} max={stats['max']:.2e}")

    def remove_hooks(self):

        for hook in self.hooks:

            hook.remove()

        self.hooks.clear()

In [ ]:
```

### Step 2: The Overfit-One-Batch Test

In [ ]:
```python

def overfit_one_batch(model, x_batch, y_batch, criterion, lr=0.01, steps=200):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    print("\n=== OVERFIT ONE BATCH TEST ===")

    print(f"Batch size: {x_batch.shape[0]}, Steps: {steps}")

    for step in range(steps):

        optimizer.zero_grad()

        output = model(x_batch)

        loss = criterion(output, y_batch)

        loss.backward()

        optimizer.step()

        if step % 50 == 0 or step == steps - 1:

            with torch.no_grad():

                preds = (output > 0).float() if output.shape[-1] == 1 else output.argmax(dim=1)

                targets = y_batch if y_batch.dim() == 1 else y_batch.squeeze()

                acc = (preds.squeeze() == targets).float().mean().item()

            print(f"  Step {step:3d} | Loss: {loss.item():.6f} | Accuracy: {acc:.1%}")

    final_loss = loss.item()

    if final_loss > 0.1:

        print(f"\n  FAIL: Loss did not converge ({final_loss:.4f}). Model or training loop is broken.")

        return False

    print(f"\n  PASS: Loss converged to {final_loss:.6f}")

    return True

In [ ]:
```

### Step 3: Learning Rate Finder

In [ ]:
```python

def find_learning_rate(model, x_data, y_data, criterion, start_lr=1e-7, end_lr=10, steps=100):

    import copy

    original_state = copy.deepcopy(model.state_dict())

    optimizer = torch.optim.SGD(model.parameters(), lr=start_lr)

    lr_mult = (end_lr / start_lr) ** (1 / steps)

    model.train()

    results = []

    best_loss = float("inf")

    current_lr = start_lr

    print("\n=== LEARNING RATE FINDER ===")

    for step in range(steps):

        optimizer.zero_grad()

        output = model(x_data)

        loss = criterion(output, y_data)

        if math.isnan(loss.item()) or loss.item() > best_loss * 10:

            break

        best_loss = min(best_loss, loss.item())

        results.append((current_lr, loss.item()))

        loss.backward()

        optimizer.step()

        current_lr *= lr_mult

        for param_group in optimizer.param_groups:

            param_group["lr"] = current_lr

    model.load_state_dict(original_state)

    if len(results) < 10:

        print("  Could not complete LR sweep -- loss diverged too quickly")

        return results

    min_loss_idx = min(range(len(results)), key=lambda i: results[i][1])

    suggested_lr = results[max(0, min_loss_idx - 10)][0]

    print(f"  Swept {len(results)} steps from {start_lr:.0e} to {results[-1][0]:.0e}")

    print(f"  Minimum loss {results[min_loss_idx][1]:.4f} at lr={results[min_loss_idx][0]:.2e}")

    print(f"  Suggested learning rate: {suggested_lr:.2e}")

    return results

In [ ]:
```

### Step 4: Gradient Checker

In [ ]:
```python

def _flat_to_multi_index(flat_idx, shape):

    multi_idx = []

    remaining = flat_idx

    for dim in reversed(shape):

        multi_idx.insert(0, remaining % dim)

        remaining //= dim

    return tuple(multi_idx)

def gradient_check(model, x, y, criterion, eps=1e-4):

    model.train()

    x_double = x.double()

    y_double = y.double()

    model_double = model.double()

    print("\n=== GRADIENT CHECK ===")

    overall_max_diff = 0

    checked = 0

    for name, param in model_double.named_parameters():

        if not param.requires_grad:

            continue

        layer_max_diff = 0

        model_double.zero_grad()

        output = model_double(x_double)

        loss = criterion(output, y_double)

        loss.backward()

        analytical_grad = param.grad.clone()

        num_checks = min(5, param.numel())

        for i in range(num_checks):

            idx = _flat_to_multi_index(i, param.shape)

            original = param.data[idx].item()

            param.data[idx] = original + eps

            with torch.no_grad():

                loss_plus = criterion(model_double(x_double), y_double).item()

            param.data[idx] = original - eps

            with torch.no_grad():

                loss_minus = criterion(model_double(x_double), y_double).item()

            param.data[idx] = original

            numerical = (loss_plus - loss_minus) / (2 * eps)

            analytical = analytical_grad[idx].item()

            denom = max(abs(numerical), abs(analytical), 1e-8)

            rel_diff = abs(numerical - analytical) / denom

            layer_max_diff = max(layer_max_diff, rel_diff)

            checked += 1

        overall_max_diff = max(overall_max_diff, layer_max_diff)

        status = "OK" if layer_max_diff < 1e-5 else "MISMATCH"

        print(f"  {name}: max_rel_diff={layer_max_diff:.2e} [{status}]")

    model.float()

    print(f"\n  Checked {checked} parameters")

    if overall_max_diff < 1e-5:

        print("  PASS: Gradients match (rel_diff < 1e-5)")

    elif overall_max_diff < 1e-3:

        print("  WARN: Small differences (1e-5 < rel_diff < 1e-3)")

    else:

        print("  FAIL: Gradient mismatch detected (rel_diff > 1e-3)")

    return overall_max_diff

In [ ]:
```

### Step 5: Deliberately Broken Networks

Now apply the toolkit to broken networks and diagnose each one.

In [ ]:
```python

def demo_broken_networks():

    torch.manual_seed(42)

    x = torch.randn(64, 10)

    y = (x[:, 0] > 0).long()

    print("\n" + "=" * 60)

    print("BUG 1: Learning rate too high (lr=10)")

    print("=" * 60)

    model1 = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

    debugger1 = NetworkDebugger(model1)

    optimizer1 = torch.optim.SGD(model1.parameters(), lr=10.0)

    criterion = nn.CrossEntropyLoss()

    for step in range(20):

        optimizer1.zero_grad()

        out = model1(x)

        loss = criterion(out, y)

        debugger1.record_loss(loss.item())

        loss.backward()

        optimizer1.step()

    debugger1.print_report()

    debugger1.remove_hooks()

    print("\n" + "=" * 60)

    print("BUG 2: Dead ReLUs from bad initialization")

    print("=" * 60)

    model2 = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))

    with torch.no_grad():

        for m in model2.modules():

            if isinstance(m, nn.Linear):

                m.weight.fill_(-1.0)

                m.bias.fill_(-5.0)

    debugger2 = NetworkDebugger(model2)

    optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

    for step in range(50):

        optimizer2.zero_grad()

        out = model2(x)

        loss = criterion(out, y)

        debugger2.record_loss(loss.item())

        loss.backward()

        optimizer2.step()

    debugger2.print_report()

    debugger2.remove_hooks()

    print("\n" + "=" * 60)

    print("BUG 3: Missing zero_grad (gradients accumulate)")

    print("=" * 60)

    model3 = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

    debugger3 = NetworkDebugger(model3)

    optimizer3 = torch.optim.SGD(model3.parameters(), lr=0.01)

    for step in range(50):

        out = model3(x)

        loss = criterion(out, y)

        debugger3.record_loss(loss.item())

        loss.backward()

        optimizer3.step()

    debugger3.print_report()

    debugger3.remove_hooks()

    print("\n" + "=" * 60)

    print("HEALTHY NETWORK: Correct setup for comparison")

    print("=" * 60)

    model_good = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

    debugger_good = NetworkDebugger(model_good)

    optimizer_good = torch.optim.Adam(model_good.parameters(), lr=1e-3)

    for step in range(50):

        optimizer_good.zero_grad()

        out = model_good(x)

        loss = criterion(out, y)

        debugger_good.record_loss(loss.item())

        loss.backward()

        optimizer_good.step()

    debugger_good.print_report()

    debugger_good.remove_hooks()

    print("\n" + "=" * 60)

    print("OVERFIT-ONE-BATCH TEST (healthy model)")

    print("=" * 60)

    model_test = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

    overfit_one_batch(model_test, x[:8], y[:8], criterion)

    print("\n" + "=" * 60)

    print("LEARNING RATE FINDER")

    print("=" * 60)

    model_lr = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

    find_learning_rate(model_lr, x, y, criterion)

    print("\n" + "=" * 60)

    print("GRADIENT CHECK")

    print("=" * 60)

    model_grad = nn.Sequential(nn.Linear(10, 8), nn.ReLU(), nn.Linear(8, 2))

    gradient_check(model_grad, x[:4], y[:4], criterion)

In [ ]:
```

## Exercises

In [ ]:
1. **Add an exploding gradient detector.** Modify the `NetworkDebugger` to detect when gradients exceed a threshold and automatically suggest a gradient clipping value. Test it on a 20-layer network with no normalization.

2. **Build a dead neuron resurrector.** Write a function that identifies dead ReLU neurons (always outputting 0) and reinitializes their incoming weights with Kaiming initialization. Show that this recovers a network where >70% of neurons are dead.

3. **Implement the learning rate finder with plotting.** Extend `find_learning_rate` to save results as a CSV and write a separate script that reads the CSV and displays the LR vs loss curve using matplotlib. Identify the optimal LR for ResNet-18 on CIFAR-10.

4. **Create a data pipeline validator.** Write a function that checks for: duplicate samples across train/test splits, label distribution imbalance (>10:1 ratio), input normalization (mean near 0, std near 1), and NaN/Inf values in the data. Run it on a deliberately corrupted dataset.

5. **Debug a real failure.** Take the mini-framework from Lesson 10, introduce a subtle bug (e.g., transpose the weight matrix in backward), and use gradient checking to locate exactly which parameter has incorrect gradients. Document the debugging process.